# OmniSearch — Setup & Hello World 🔥🤖

**MIDS Capstone · Summer 2026 · UC Berkeley**

MAPPO/HAPPO-trained drone + ground robot swarms for wildfire survivor search.

## What this notebook does

1. **Check the environment** — Python version, GPU availability
2. **(Optional) Install dependencies** — PyTorch, VMAS, TorchRL, BenchMARL, YOLOv8, W&B
3. **Verify** each component imports and works
4. **Run a hello-world VMAS demo** — 5 agents in `navigation`, 100 random steps across 32 parallel envs
5. **Preview the BenchMARL training API**

## Prerequisites

Create a Python 3.10 or 3.11 environment (venv or conda) and select its kernel in Jupyter before running:

```bash
python3.11 -m venv .venv
source .venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name omnisearch --display-name "Python (omnisearch)"
```

## 1. Environment check

In [1]:
import sys, platform
print(f"Python:    {sys.version.split()[0]}")
print(f"Platform:  {platform.platform()}")
print(f"Executable: {sys.executable}")
assert sys.version_info[:2] in [(3, 10), (3, 11)], \
    "OmniSearch needs Python 3.10 or 3.11 (VMAS / BenchMARL compatibility)."

Python:    3.11.14
Platform:  macOS-26.3.1-arm64-arm-64bit
Executable: /Users/alexlavre/Documents/omnisearch_capstone/omnisearch/.venv/bin/python


## 2. (Optional) Install dependencies

Skip if you've already installed. SimFire is omitted because it requires Python <3.10 (PyGame constraint) — the project uses a cellular-automata fallback for fire spread.

For CUDA-specific PyTorch, see https://pytorch.org/get-started/locally/.

In [ ]:
# Uncomment to install. Comment back out once done — runs are slow.

# %pip install --upgrade pip
# %pip install torch torchvision
# %pip install vmas torchrl benchmarl ultralytics "pettingzoo[mpe]"
# %pip install wandb tensorboard tqdm hydra-core omegaconf matplotlib seaborn pandas numpy pyyaml
# %pip install pytest black ruff ipykernel "moviepy<2.0.0"

# SimFire (will fail on Python 3.11 — that's fine, we use a fallback fire model):
# %pip install simfire

## 3. Verify each component

Each cell tests one piece of the stack. Run them in order — if any fail, fix that one before proceeding.

In [3]:
# --- PyTorch ---
import torch
cuda = torch.cuda.is_available()
info = f"CUDA {torch.version.cuda} ({torch.cuda.get_device_name(0)})" if cuda else "CPU only"
print(f"✓ PyTorch {torch.__version__} | {info}")

✓ PyTorch 2.12.0 | CPU only


In [4]:
# --- VMAS (Vectorized Multi-Agent Simulator) ---
import vmas
env = vmas.make_env(
    scenario="navigation",
    num_envs=4,
    n_agents=3,         # scenario kwarg — passed via **kwargs to make_world
    device="cpu",
)
obs = env.reset()
print(f"✓ VMAS {getattr(vmas, '__version__', '?')} | navigation env, 3 agents, 4 parallel envs")
print(f"  Observation shape (per agent): {obs[0].shape}")

✓ VMAS 1.5.2 | navigation env, 3 agents, 4 parallel envs
  Observation shape (per agent): torch.Size([4, 18])


In [5]:
# --- TorchRL ---
import torchrl
from torchrl.envs.libs.vmas import VmasEnv
print(f"✓ TorchRL {torchrl.__version__} | VmasEnv wrapper available")

✓ TorchRL 0.11.1 | VmasEnv wrapper available


In [6]:
# --- BenchMARL ---
import benchmarl
from benchmarl.algorithms import MappoConfig
print(f"✓ BenchMARL installed | MappoConfig available")
print("  Algorithms: MAPPO, IPPO, QMIX, MADDPG, VDN, IQL, ...")

✓ BenchMARL installed | MappoConfig available
  Algorithms: MAPPO, IPPO, QMIX, MADDPG, VDN, IQL, ...


In [7]:
# --- YOLOv8 (person detector for survivors) ---
from ultralytics import YOLO
model = YOLO("yolov8n.pt")  # auto-downloads 6MB nano model on first run
assert model.names[0] == "person", f"Expected class 0 = person, got {model.names[0]!r}"
print(f"✓ YOLOv8 | yolov8n.pt | 80 COCO classes")
print(f"  Class 0 = {model.names[0]!r} ← survivor detector")

✓ YOLOv8 | yolov8n.pt | 80 COCO classes
  Class 0 = 'person' ← survivor detector


In [8]:
# --- Weights & Biases ---
import wandb
print(f"✓ W&B {wandb.__version__}")
print("  Run `wandb login` in a shell to authenticate before training.")

✓ W&B 0.27.0
  Run `wandb login` in a shell to authenticate before training.


## 4. Hello-world demo: 5 agents in VMAS `navigation`

Random actions for 100 steps across 32 parallel envs. The reward will be poor (random policy) — training will fix that.

Conceptually these 5 agents map to **3 drones + 2 ground robots**; the custom `WildfireSearchScenario` in [`../envs/wildfire_search.py`](../envs/wildfire_search.py) will replace `navigation` with heterogeneous agent types, survivor landmarks, and a spreading fire.

In [9]:
import torch, vmas

device  = "cuda" if torch.cuda.is_available() else "cpu"
n_agents = 5
n_envs   = 32

env = vmas.make_env(
    scenario="navigation",
    num_envs=n_envs,
    n_agents=n_agents,
    device=device,
    continuous_actions=True,
)

print(f"Device: {device}")
print(f"Agents: {n_agents} (conceptually: 3 drones + 2 ground robots)")
print(f"Parallel envs: {n_envs}")
print(f"Obs space:    {env.observation_space}")
print(f"Action space: {env.action_space}")

Device: cpu
Agents: 5 (conceptually: 3 drones + 2 ground robots)
Parallel envs: 32
Obs space:    Tuple(Box(-inf, inf, (18,), float32), Box(-inf, inf, (18,), float32), Box(-inf, inf, (18,), float32), Box(-inf, inf, (18,), float32), Box(-inf, inf, (18,), float32))
Action space: Tuple(Box(-1.0, 1.0, (2,), float32), Box(-1.0, 1.0, (2,), float32), Box(-1.0, 1.0, (2,), float32), Box(-1.0, 1.0, (2,), float32), Box(-1.0, 1.0, (2,), float32))


In [10]:
obs = env.reset()
total_reward = torch.zeros(n_envs, device=device)

for step in range(100):
    # VMAS helper — samples within each agent's valid action range.
    # (torch.randn would violate the range assertion in env.step.)
    actions = env.get_random_actions()
    obs, rewards, dones, infos = env.step(actions)
    total_reward += sum(r.squeeze() for r in rewards)

print(f"✓ 100 steps completed")
print(f"  Average total reward across {n_envs} envs: {total_reward.mean().item():.2f}")
print("  (Random policy — training will improve this dramatically.)")

✓ 100 steps completed
  Average total reward across 32 envs: -3.04
  (Random policy — training will improve this dramatically.)


## 5. BenchMARL training preview

Once the custom wildfire scenario is wired up, training MAPPO/HAPPO is a one-liner.

**CLI:**
```bash
python -m benchmarl.run \
  algorithm=mappo \
  task=vmas/navigation \
  experiment.max_n_iters=100
```

**Python:**
```python
from benchmarl.algorithms import MappoConfig
from benchmarl.environments import VmasTask
from benchmarl.experiment import Experiment, ExperimentConfig

experiment = Experiment(
    algorithm_config=MappoConfig.get_from_yaml(),
    task=VmasTask.NAVIGATION.get_from_yaml(),
    seed=0,
    config=ExperimentConfig.get_from_yaml(),
)
experiment.run()
```

## Next steps

1. Flesh out [`WildfireSearchScenario`](../envs/wildfire_search.py) — heterogeneous agents (Drone + DiffDrive dynamics), survivor landmarks, fire spread
2. Integrate a fire model (cellular automata on Python 3.11; SimFire requires a separate 3.10 env)
3. Wire the scenario into BenchMARL as a custom `VmasTask`
4. Train HAPPO and benchmark vs. MAPPO / IPPO baselines
5. Add the comms-dropout ablation

## References

- VMAS: https://github.com/proroklab/VectorizedMultiAgentSimulator
- BenchMARL: https://github.com/facebookresearch/BenchMARL
- TorchRL: https://github.com/pytorch/rl
- SimFire: https://github.com/mitrefireline/simfire
- YOLOv8: https://github.com/ultralytics/ultralytics
- MAPPO paper: https://arxiv.org/abs/2103.01955
- HAPPO paper: https://arxiv.org/abs/2304.09870